# SHViT + Oxford Pets on Google Colab

This notebook walks through:
1. Enabling GPU and checking the environment
2. Cloning SHViT and installing dependencies
3. Downloading pretrained SHViT-S4 weights
4. Downloading Oxford Pets with Tip-Adapter style preprocessing
5. Verifying the model loads and runs inference
6. Running SHViT's official eval script

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`
> All outputs from this stage are saved under `/content/CV_Research_Paper_OxfordPets/`.


## 0. Check GPU & environment

In [1]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import sys
print('Python version  :', sys.version.split()[0])

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 102.0 GB
Python version  : 3.12.13


## 1. (Optional) Mount Google Drive

Oxford Pets is small (~800 MB, 7,349 images). Mounting Drive lets you
keep the unpacked dataset and downloaded weights between Colab restarts.
Skip this cell if you are happy to re-download every session.


In [2]:
USE_DRIVE = False   # set True to persist data + outputs in Google Drive

OUT_ROOT = '/content/CV_Research_Paper_OxfordPets'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/oxford_pets_data'
    OUT_ROOT  = '/content/drive/MyDrive/CV_Research_Paper_OxfordPets'
else:
    DATA_ROOT = '/content/oxford_pets_data'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('Dataset will be stored at:', DATA_ROOT)
print('Outputs will be written under:', OUT_ROOT)


Dataset will be stored at: /content/oxford_pets_data
Outputs will be written under: /content/CV_Research_Paper_OxfordPets


## 2. Clone SHViT

In [3]:
import os

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT
else:
    print('SHViT already cloned, skipping.')

!ls /content/SHViT

Cloning into '/content/SHViT'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 183 (delta 84), reused 82 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 168.52 KiB | 6.24 MiB/s, done.
Resolving deltas: 100% (84/84), done.
acc_vs_thro.png  engine.py	  losses.py  README.md	       utils.py
data		 export_model.py  main.py    requirements.txt
downstream	 LICENSE	  model      speed_test.py


## 3. Install dependencies

Colab ships with PyTorch 2.x which satisfies SHViT's `>=1.11` requirement,
so we only need to install the extra packages from `requirements.txt`.

`--no-deps` on timm avoids overwriting Colab's torch/torchvision with
the older versions timm 0.5.4 would otherwise pull in.

In [4]:
# scikit-image==0.19.3 from SHViT's requirements has no wheels for Python
# 3.12 (Colab's default) — and we don't actually need it. Install only what
# the SHViT model architecture needs.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 20.9 MB/s eta 0:00:00
Dependencies installed.


## 4. Download SHViT-S4 pretrained weights

In [5]:
WEIGHTS_DIR = '/content/weights'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
else:
    print('Weights already downloaded, skipping.')

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint size: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

/content/weights/sh 100%[===================>] 254.37M   352MB/s    in 0.7s    
Checkpoint size: 266.7 MB  ->  /content/weights/shvit_s4.pth


## 5. Download Oxford Pets with Tip-Adapter style preprocessing

This clones our Oxford Pets project and runs `prepare_oxford_pets.py`, which
downloads the dataset via `torchvision.datasets.OxfordPets` and prepares a
Tip-Adapter style `split_zhou_OxfordPets.json` (using gdown if available,
otherwise generating a deterministic 50/20/30 train/val/test split).


In [6]:
import os, shutil

REPO_DIR = '/content/Vision_Project_spring_26'
if not os.path.isdir(REPO_DIR):
    !git clone -b Vision_Project_spring_26_OxfordPets \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Make sure the prepare script + dataset helpers are importable from /content
for fname in [
    'prepare_oxford_pets.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{fname}')

# Vendor the datasets/ package (Tip-Adapter utilities)
DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO_DIR}/datasets', DATASETS_DST)

!pip install -q gdown
!python /content/prepare_oxford_pets.py --root {DATA_ROOT}


Cloning into '/content/Vision_Project_spring_26'...
remote: Enumerating objects: 663, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 663 (delta 90), reused 88 (delta 88), pack-reused 570 (from 1)
Receiving objects: 100% (663/663), 444.08 MiB | 42.78 MiB/s, done.
Resolving deltas: 100% (333/333), done.
[oxford_pets] downloading Oxford-IIIT Pet via torchvision into /content/oxford_pets_data ...
100% 792M/792M [00:03<00:00, 204MB/s]
100% 19.2M/19.2M [00:00<00:00, 132MB/s] 
[oxford_pets] moving /content/oxford_pets_data/oxford-iiit-pet/images -> /content/oxford_pets_data/oxford_pets/images
[oxford_pets] moving /content/oxford_pets_data/oxford-iiit-pet/annotations -> /content/oxford_pets_data/oxford_pets/annotations
[oxford_pets] trying official split download via gdown: https://drive.google.com/uc?id=1501r8Ber4nNKvmlFVQZ8SeUHTcdTTZkw
[oxford_pets] official split download failed (Failed to retrieve file url:

	Cannot retrieve

In [7]:
# Sanity check
from pathlib import Path
ds_root = Path(DATA_ROOT) / 'oxford_pets'
img_dir = ds_root / 'images'
split_json = ds_root / 'split_zhou_OxfordPets.json'

n_img = sum(1 for _ in img_dir.glob('*.jpg')) if img_dir.exists() else 0
print(f'On-disk:  {n_img} images at {img_dir}')
print(f'Split    : {split_json} (exists: {split_json.exists()})')


On-disk:  7390 images at /content/oxford_pets_data/oxford_pets/images
Split    : /content/oxford_pets_data/oxford_pets/split_zhou_OxfordPets.json (exists: True)


## 6. Verify model loads and runs inference

Loads the SHViT-S4 checkpoint and runs 50 Oxford Pets images through it.
Predictions are ImageNet class indices (not Oxford Pets labels) — accuracy
will be ~zero until the model is fine-tuned. The goal here is just to
confirm no import / shape errors occur.


In [8]:
import sys, time, pathlib
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

sys.path.insert(0, '/content/SHViT')

from model import shvit
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

model_shvit = timm.create_model('shvit_s4', pretrained=False, num_classes=1000)

ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('model', ckpt)
missing, unexpected = model_shvit.load_state_dict(state_dict, strict=False)
print(f'Missing keys: {len(missing)}   Unexpected keys: {len(unexpected)}')

model_shvit.to(DEVICE).eval()
print('Model loaded successfully.')


Using device: cuda
Missing keys: 0   Unexpected keys: 0
Model loaded successfully.


In [9]:
NUM_IMAGES = 50

# CLIP / Tip-Adapter normalization
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

# Oxford Pets ships all images in a single flat images/ folder.
# Breed is encoded in the filename, e.g. "Abyssinian_1.jpg".
image_dir = pathlib.Path(DATA_ROOT) / 'oxford_pets' / 'images'
items = []
for img_path in sorted(image_dir.glob('*.jpg'))[:NUM_IMAGES]:
    breed = '_'.join(img_path.stem.split('_')[:-1]) or 'unknown'
    items.append((img_path, breed))

print(f'Running inference on {len(items)} images ...')
t0 = time.perf_counter()
results = []
with torch.no_grad():
    for img_path, true_class in items:
        x = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = int(model_shvit(x).argmax(1).item())
        results.append((img_path.name, true_class, pred))

elapsed = time.perf_counter() - t0
print(f'\n{"Image":<30} {"True class":<25} {"Pred idx":>8}')
print('-' * 65)
for name, cls, pred in results[:15]:
    print(f'{name:<30} {cls:<25} {pred:>8}')
print(f'\nTotal: {elapsed:.2f}s  ({elapsed/len(results)*1000:.1f} ms/image)')
print('\n[OK] Model ran without errors.')


Running inference on 50 images ...

Image                          True class                Pred idx
-----------------------------------------------------------------
Abyssinian_1.jpg               Abyssinian                     151
Abyssinian_10.jpg              Abyssinian                     285
Abyssinian_100.jpg             Abyssinian                     728
Abyssinian_101.jpg             Abyssinian                     285
Abyssinian_102.jpg             Abyssinian                     285
Abyssinian_103.jpg             Abyssinian                     285
Abyssinian_104.jpg             Abyssinian                     358
Abyssinian_105.jpg             Abyssinian                     285
Abyssinian_106.jpg             Abyssinian                     281
Abyssinian_107.jpg             Abyssinian                     285
Abyssinian_108.jpg             Abyssinian                     285
Abyssinian_109.jpg             Abyssinian                     273
Abyssinian_11.jpg              Abyssinia

## 7. Run SHViT's official eval script

Oxford Pets stores every image in one flat `images/` folder, so we first
build a temporary per-class symlink tree from the Tip-Adapter split JSON;
SHViT's `--data-set IMNET` then accepts it as an ImageFolder-style dataset.
Expect ~zero accuracy relative to ImageNet-1K classes — fine-tuning happens
in Stage 3.


In [10]:
import re, json, os, shutil

with open('/content/SHViT/main.py', 'r') as f:
    content = f.read()
content = re.sub(r"torch\.load\(([^,]+),\s*map_location='cpu'\)",
                 r"torch.load(\1, map_location='cpu', weights_only=False)",
                 content)
with open('/content/SHViT/main.py', 'w') as f:
    f.write(content)

# Build a per-class symlink tree from the Tip-Adapter split JSON so that
# SHViT's IMNET data-set finds an ImageFolder-style layout.
IM_ROOT = '/content/imnet_oxford_pets_sanity'
SPLIT_JSON = f'{DATA_ROOT}/oxford_pets/split_zhou_OxfordPets.json'
IMG_ROOT   = f'{DATA_ROOT}/oxford_pets/images'

with open(SPLIT_JSON) as f:
    split = json.load(f)

if os.path.isdir(IM_ROOT):
    shutil.rmtree(IM_ROOT)
for tgt in ('train', 'val'):
    for rel_path, label, classname in split[tgt]:
        cls_safe = classname.replace(' ', '_').replace('/', '_')
        cls_dir = f'{IM_ROOT}/{tgt}/{cls_safe}'
        os.makedirs(cls_dir, exist_ok=True)
        src = f'{IMG_ROOT}/{rel_path}'
        dst = f'{cls_dir}/{os.path.basename(rel_path)}'
        if not os.path.exists(dst):
            os.symlink(src, dst)

!python /content/SHViT/main.py \
    --model shvit_s4 \
    --eval \
    --resume {WEIGHTS_PATH} \
    --data-path {IM_ROOT} \
    --data-set IMNET \
    --batch-size 64 \
    --num_workers 2 \
    --device cuda


Not using distributed mode
Creating model: shvit_s4
number of params: 16588484
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Loading local checkpoint at /content/weights/shvit_s4.pth
<All keys matched successfully>
Evaluating model: shvit_s4
/content/SHViT/engine.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Test:  [ 0/16]  eta: 0:02:09  loss: 8.1486 (8.1486)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 8.0908  data: 0.3258  max mem: 479
Test:  [10/16]  eta: 0:00:04  loss: 8.3718 (8.3455)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 0.8174  data: 0.1044  max mem: 479
Test:  [15/16]  eta: 0:00:01  loss: 8.3250 (8.3618)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time

## Next steps — fine-tuning on Oxford Pets

To actually train SHViT on Oxford Pets, head over to Stage 3 and run
`finetune_shvit_oxford_pets.py`, which uses the Tip-Adapter split JSON,
CLIP normalization, RandAugment / RandomErasing / Mixup / CutMix /
label-smoothing, AGC-style gradient clipping, and cosine LR with warmup
(the same recipe as the SHViT paper, but with `--nb_classes 37`).
